# 🔐 LAB 4A — Implement RBAC-Enforced RAG
> **Block 4 | 20 Minutes | Platform: HPE Private Cloud AI**

---

## What This Lab Is About

In Lab 3A you analysed a query log to identify content gaps — you saw how retrieval scores vary across topic clusters and what happens when the corpus does not cover a user’s question. This lab shifts focus from *what* the system retrieves to *who* is allowed to retrieve it.

Enterprise RAG systems must enforce access control at the document level. A guest user must never see executive compensation data. An employee must not retrieve confidential finance reports. The challenge is that Qdrant — like all major vector stores — has no native concept of users or roles. It executes whatever filter your application sends it.

This lab builds a complete **application-layer RBAC pipeline**: JWT tokens carry signed role claims, your code decodes and verifies them, constructs a Qdrant payload filter from the resolved permissions, and injects that filter into every retrieval call. The vector store never sees a role — it only sees a pre-built filter.

The output of this lab is a working RBAC-enforced RAG chain with a validated audit log proving that access boundaries held across all four roles.

---

## 🎯 Learning Objectives

- Ingest documents with structured RBAC metadata tags into Qdrant
- Understand where RBAC is enforced (application layer, not Qdrant server)
- Build a JWT-validated metadata-filtered retriever
- Verify access boundaries with cross-role query testing
- Validate audit log entries for compliance

---

## 🗺️ Lab Structure

| Step | Cell | What You Are Building | Target Time |
|---|---|---|---|
| Config | 1 | Endpoints, credentials, paths, parameters | 1 min |
| Init | 2 | LLM, embeddings, Langfuse, Qdrant clients | 2 min |
| Step 1 | 3–4 | Ingest 12 documents with RBAC metadata into Qdrant | 3 min |
| Step 2 | 5 | Configure role permission profiles | 2 min |
| Step 3 | 6 | JWT decoder + RBAC retriever builder | 4 min |
| Step 4 | 7–8 | Cross-role query execution + side-by-side comparison | 3 min |
| Step 5 | 9 | Access boundary verification per chunk | 3 min |
| Step 6 | 10 | Audit log review and compliance validation | 1 min |
| Validate | 11 | Run full validation suite | 1 min |

> ⚠️ Past 20 minutes and stuck? Open `04_solution.ipynb` — all cells are pre-run.

---

## 📁 Data Source: RBAC Documents

| Item | Detail |
|---|---|
| **Document count** | 12 mixed-classification `.txt` files |
| **Metadata sidecar** | Paired `.meta.json` per document |
| **Levels** | 1=Public, 2=Internal, 3=Confidential, 4=Restricted |
| **JWT tokens** | 4 pre-signed tokens in `/data/workshop/tokens/` |
| **Docs path** | `/data/workshop/rbac_docs/` |

---

## 🏗 RBAC Architecture

Before writing any code, understand **where each security boundary lives**.

```
╔══════════════════════════════════════════════════════════════════════╗
║                    RBAC-ENFORCED RAG — ARCHITECTURE                  ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║   USER / CLIENT                                                      ║
║   ┌─────────────────┐                                                ║
║   │  JWT Token      │  ← Pre-issued by auth system                  ║
║   │  (signed)       │    Contains: user_id, role, dept              ║
║   └────────┬────────┘                                                ║
║            │  token only — role never sent as plaintext             ║
║            ▼                                                         ║
║   ┌─────────────────────────────────────────────────────────┐       ║
║   │            APPLICATION LAYER  (this notebook)           │       ║
║   │                                                         │       ║
║   │  ① decode_jwt(token)                                    │       ║
║   │       └─► verify signature with JWT_SECRET             │       ║
║   │           extract: role, user_id, dept                 │       ║
║   │                                                         │       ║
║   │  ② get_permissions(role)                                │       ║
║   │       └─► lookup ROLE_PERMISSIONS dict                 │       ║
║   │           resolve: max_access_level, allowed_depts     │       ║
║   │                                                         │       ║
║   │  ③ build_qdrant_filter(max_level, allowed_depts)        │       ║
║   │       └─► Filter(must=[                                │       ║
║   │               access_level <= max_level,               │       ║
║   │               department IN allowed_depts              │       ║
║   │           ])                                            │       ║
║   │                                                         │       ║
║   │  ④ retriever.invoke(query, filter=qdrant_filter)        │       ║
║   │                                                         │       ║
║   │  ⑤ audit_log(user, decision, filter, chunks_count)      │       ║
║   │                                                         │       ║
║   └───────────────────────────┬─────────────────────────┘       ║
║                               │  ANN search + payload filter        ║
║                               ▼                                      ║
║   ┌─────────────────────────────────────────────────────────┐       ║
║   │            QDRANT SERVER  (external, no API key)        │       ║
║   │                                                         │       ║
║   │  Receives : query_vector + Filter object                │       ║
║   │  Executes : ANN search constrained by payload filter    │       ║
║   │  Returns  : matching vectors + payloads                 │       ║
║   │                                                         │       ║
║   │  ⚠ Qdrant does NOT know about roles or users.          │       ║
║   │    It only executes the filter it receives.             │       ║
║   │    Security depends entirely on the app layer above.   │       ║
║   │                                                         │       ║
║   └─────────────────────────────────────────────────────────┘       ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  RBAC ENFORCEMENT BOUNDARY:  ^^^^ APPLICATION LAYER ^^^^            ║
║  Qdrant role:  Execute pre-built filters faithfully                  ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 🔑 Role → Filter Mapping

```
  ROLE        MAX_LEVEL   DEPARTMENTS              QDRANT FILTER PRODUCED
  ──────────  ─────────   ──────────────────────   ────────────────────────────────────────
  admin           4       [*] all                  access_level <= 4
  manager         3       [finance,hr,engineering] access_level <= 3 AND dept IN [...]
  employee        2       [engineering,product]    access_level <= 2 AND dept IN [...]
  guest           1       [public]                 access_level <= 1 AND dept IN [public]
```

---

## 📦 Document Classification Schema

```
  Each vector point stored in Qdrant carries this payload:
  ┌─────────────────────────────────────────────────────┐
  │  {                                                  │
  │    "access_level":   int,   # 1=Public  4=Restrict  │
  │    "department":     str,   # finance / hr / etc.   │
  │    "classification": str,   # public/internal/...   │
  │    "owner":          str,   # team identifier       │
  │    "ingested_at":    str,   # ISO 8601 datetime     │
  │    "source_file":    str,   # origin filename       │
  │  }                                                  │
  └─────────────────────────────────────────────────────┘
  These payload fields are what Qdrant filters against.
  They are set at INGEST time and never modified by users.
```

## ⚙️ Cell 1 — Configuration

All tunable parameters and endpoint credentials are defined here. No environment files are used — fill in every value explicitly before running. The random seed ensures reproducible results across runs.

TLS verification is disabled via `httpx.Client(verify=False)` — required for HPE Private Cloud AI self-signed certificates.

In [ ]:
# ============================================================
# CELL 1 — Configuration
# ✏️  Fill in endpoint credentials before running.
# ============================================================
import warnings
warnings.filterwarnings("ignore")

# --- HPE Private Cloud AI — LLM endpoint -------------------------
LLM_BASE_URL   = "https://your-hpe-pcai-host/v1"          # ✏️ replace
LLM_API_KEY    = "your-llm-api-key"                       # ✏️ replace
LLM_MODEL_NAME = "mistral-7b-instruct"                    # ✏️ replace

# --- HPE Private Cloud AI — Embedding endpoint -------------------
EMBED_BASE_URL   = "https://your-hpe-pcai-host/v1"        # ✏️ replace
EMBED_API_KEY    = "your-embed-api-key"                   # ✏️ replace
EMBED_MODEL_NAME = "nvidia/nv-embedqa-mistral-7b-v2"      # ✏️ replace
EMBED_DIM        = 4096                                   # ✏️ match your model
EMBED_BATCH      = 50

# --- Qdrant — external, no API key -------------------------------
QDRANT_HOST       = "your-qdrant-host"                    # ✏️ replace
QDRANT_PORT       = 6333                                  # ✏️ replace
QDRANT_USE_HTTPS  = False                                 # ✏️ True if HTTPS
QDRANT_COLLECTION = "rbac_workshop"

# --- Langfuse — external observability ---------------------------
LANGFUSE_PUBLIC_KEY = "pk-lf-xxxxxxxxxxxxxxxxxxxx"        # ✏️ replace
LANGFUSE_SECRET_KEY = "sk-lf-xxxxxxxxxxxxxxxxxxxx"        # ✏️ replace
LANGFUSE_HOST       = "https://your-langfuse-host"        # ✏️ replace

# --- Workshop paths ----------------------------------------------
DOCS_DIR   = "/data/workshop/rbac_docs"
TOKENS_DIR = "/data/workshop/tokens"

# --- JWT signing -------------------------------------------------
JWT_SECRET = "hpe-workshop-secret-2024"
JWT_ALGO   = "HS256"

# --- RBAC thresholds ---------------------------------------------
MISS_THRESHOLD = 0.50
N_GAP_CLUSTERS = 3
SAMPLE_QUERIES = 10
DENIED_MESSAGE = "You don't have permission to access this."

# --- Reproducibility ---------------------------------------------
RANDOM_SEED = 42

import numpy as np
np.random.seed(RANDOM_SEED)

# --- TLS — skip verification for HPE PCAI self-signed certs ------
import httpx
http_client = httpx.Client(verify=False)

# --- Validate credentials ----------------------------------------
unfilled = [
    k for k, v in {
        "LLM_BASE_URL"       : LLM_BASE_URL,
        "LLM_API_KEY"        : LLM_API_KEY,
        "EMBED_BASE_URL"     : EMBED_BASE_URL,
        "EMBED_API_KEY"      : EMBED_API_KEY,
        "EMBED_MODEL_NAME"   : EMBED_MODEL_NAME,
        "QDRANT_HOST"        : QDRANT_HOST,
        "LANGFUSE_PUBLIC_KEY": LANGFUSE_PUBLIC_KEY,
        "LANGFUSE_SECRET_KEY": LANGFUSE_SECRET_KEY,
        "LANGFUSE_HOST"      : LANGFUSE_HOST,
    }.items()
    if not v or "your-" in v
]
if unfilled:
    raise ValueError(
        f"\u274c Placeholder values still present: {unfilled}\n"
        f"   Fill in all endpoint credentials above."
    )

print("\u2705 Configuration loaded.")
print(f"   LLM endpoint    : {LLM_BASE_URL}")
print(f"   LLM model       : {LLM_MODEL_NAME}")
print(f"   Embed endpoint  : {EMBED_BASE_URL}")
print(f"   Embed model     : {EMBED_MODEL_NAME}")
print(f"   Embed dim       : {EMBED_DIM}d")
print(f"   Embed batch     : {EMBED_BATCH}")
print(f"   Qdrant          : {QDRANT_HOST}:{QDRANT_PORT}  (HTTPS={QDRANT_USE_HTTPS})")
print(f"   Collection      : {QDRANT_COLLECTION}")
print(f"   Langfuse        : {LANGFUSE_HOST}")
print(f"   Docs path       : {DOCS_DIR}")
print(f"   Tokens path     : {TOKENS_DIR}")
print(f"   TLS verify      : disabled (httpx.Client verify=False)")
print(f"   Random seed     : {RANDOM_SEED}")

## 🔌 Cell 2 — Initialise Clients

### Why this step exists

All external service clients are initialised once here and reused across every subsequent cell — the same pattern used in Lab 3A. The `http_client` with `verify=False` is passed explicitly to both the LLM and embedding clients to disable TLS certificate verification for HPE Private Cloud AI self-signed endpoints. Langfuse is initialised with a `CallbackHandler` that attaches to every RAG chain call automatically.

In [ ]:
# ============================================================
# CELL 2 — Initialise LLM, Embeddings, Langfuse, Qdrant
# ============================================================
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langfuse import Langfuse
from langfuse.callback import CallbackHandler
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

# --- LLM — HPE self-hosted, TLS disabled via http_client ---------
llm = ChatOpenAI(
    base_url    = LLM_BASE_URL,
    api_key     = LLM_API_KEY,
    model       = LLM_MODEL_NAME,
    temperature = 0.0,
    http_client = http_client,      # ← skips TLS verification
    timeout     = 60,
)

# --- Embeddings — HPE self-hosted, TLS disabled ------------------
embeddings = OpenAIEmbeddings(
    base_url    = EMBED_BASE_URL,
    api_key     = EMBED_API_KEY,
    model       = EMBED_MODEL_NAME,
    http_client = http_client,      # ← skips TLS verification
)

# --- Langfuse — external observability ---------------------------
langfuse_client = Langfuse(
    public_key = LANGFUSE_PUBLIC_KEY,
    secret_key = LANGFUSE_SECRET_KEY,
    host       = LANGFUSE_HOST,
)
langfuse_handler = CallbackHandler(
    public_key = LANGFUSE_PUBLIC_KEY,
    secret_key = LANGFUSE_SECRET_KEY,
    host       = LANGFUSE_HOST,
)

# --- Qdrant — external, no API key -------------------------------
qdrant_client = QdrantClient(
    host    = QDRANT_HOST,
    port    = QDRANT_PORT,
    https   = QDRANT_USE_HTTPS,
    timeout = 30,
)

if not qdrant_client.collection_exists(QDRANT_COLLECTION):
    qdrant_client.create_collection(
        collection_name = QDRANT_COLLECTION,
        vectors_config  = VectorParams(
            size     = EMBED_DIM,
            distance = Distance.COSINE,
        ),
    )
    print(f"   Collection '{QDRANT_COLLECTION}' created (dim={EMBED_DIM}).")
else:
    print(f"   Collection '{QDRANT_COLLECTION}' already exists.")

vectorstore = QdrantVectorStore(
    client          = qdrant_client,
    collection_name = QDRANT_COLLECTION,
    embedding       = embeddings,
)

print("\u2705 LLM initialised          :", LLM_MODEL_NAME)
print("\u2705 Embeddings initialised   :", EMBED_MODEL_NAME)
print("\u2705 Langfuse handler ready   :", LANGFUSE_HOST)
print("\u2705 Qdrant client ready      :", f"{QDRANT_HOST}:{QDRANT_PORT}")
print("\u2705 Vectorstore ready        :", QDRANT_COLLECTION)

---
## 🗂 Step 1 — Ingest Mixed Document Set

### Cell 3 — Load Documents with RBAC Metadata

### Why this step exists

Before any retrieval can be filtered by role, the documents must be indexed with the correct metadata. Each `.txt` file in `/data/workshop/rbac_docs/` has a paired `.meta.json` sidecar that carries the RBAC classification fields. These fields are stored as **Qdrant payload** on each vector point — they are the fields the filter will match against at query time.

This is the only moment where metadata is attached. Once indexed, users cannot modify their own payload — the access level is set by the ingest pipeline, not the requester.

```
  DOCUMENT SET OVERVIEW
  ┌──────┬──────────────┬──────────────────────┬──────────────────────┐
  │ Docs │ Level        │ Department           │ Classification       │
  ├──────┼──────────────┼──────────────────────┼──────────────────────┤
  │  3   │ 1 — Public   │ public               │ public               │
  │  3   │ 2 — Internal │ engineering, product │ internal             │
  │  3   │ 3 — Confid.  │ finance, hr, eng.    │ confidential         │
  │  3   │ 4 — Restrict │ hr, finance          │ restricted           │
  └──────┴──────────────┴──────────────────────┴──────────────────────┘
```

In [ ]:
# ============================================================
# CELL 3 — Load Documents with RBAC Metadata
# ============================================================
import json
import glob
import os
from pathlib import Path
from datetime import datetime, timezone
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

LEVEL_LABEL = {1: "PUBLIC", 2: "INTERNAL", 3: "CONFIDENTIAL", 4: "RESTRICTED"}

if not os.path.exists(DOCS_DIR):
    raise FileNotFoundError(
        f"\u274c Docs directory not found: {DOCS_DIR}\n"
        f"   Run generate_rbac_docs.py before this notebook."
    )


def load_rbac_documents(docs_dir: str) -> list:
    """
    Load .txt files with paired .meta.json sidecars.
    Metadata is attached to each Document — becomes Qdrant payload after indexing.
    Falls back to level-1 public metadata if sidecar is absent.
    """
    documents = []
    txt_files = sorted(glob.glob(f"{docs_dir}/*.txt"))

    if not txt_files:
        raise FileNotFoundError(
            f"\u274c No .txt files found in {docs_dir}\n"
            f"   Run generate_rbac_docs.py to generate the document set."
        )

    for txt_path in txt_files:
        content   = Path(txt_path).read_text(encoding="utf-8")
        meta_path = txt_path.replace(".txt", ".meta.json")
        metadata  = (
            json.loads(Path(meta_path).read_text(encoding="utf-8"))
            if Path(meta_path).exists()
            else {
                "access_level"  : 1,
                "department"    : "public",
                "classification": "public",
                "owner"         : "unknown",
                "ingested_at"   : datetime.now(timezone.utc).isoformat(),
                "source_file"   : Path(txt_path).name,
            }
        )
        documents.append(Document(page_content=content, metadata=metadata))
        print(
            f"   [{LEVEL_LABEL.get(metadata['access_level'], '?'):>12}]  "
            f"dept={metadata['department']:<12}  {Path(txt_path).name}"
        )
    return documents


print(f"Loading documents from: {DOCS_DIR}")
print("-" * 68)
raw_docs = load_rbac_documents(DOCS_DIR)
print("-" * 68)
print(f"\n   Loaded : {len(raw_docs)} documents")

# --- Split into chunks -------------------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 800,
    chunk_overlap = 100,
    separators    = ["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(raw_docs)

print(f"   Chunks after splitting : {len(chunks)}")
print()
print("   Proceed to Cell 4 — Index into Qdrant.")

### Cell 4 — Index into Qdrant and Verify Count

### Why this step exists

Indexing writes each chunk as a Qdrant **vector point** — the embedding vector plus the metadata payload. The payload fields (`access_level`, `department`, etc.) are stored alongside the vector and are what Qdrant will filter against at query time. After indexing, we verify the point count to confirm all documents were ingested correctly.

In [ ]:
# ============================================================
# CELL 4 — Index Documents into Qdrant
# ============================================================
# Reuses:
#   vectorstore       → from Cell 2
#   qdrant_client     → from Cell 2
#   chunks            → from Cell 3
#   QDRANT_COLLECTION → from Cell 1
# ============================================================

print(f"Indexing {len(chunks)} chunks into Qdrant collection '{QDRANT_COLLECTION}'...")
print(f"   Host       : {QDRANT_HOST}:{QDRANT_PORT}")
print(f"   Embed model: {EMBED_MODEL_NAME}")
print(f"   Batch size : {EMBED_BATCH}")
print()

# --- Smoke test: embed one chunk before full ingest --------------
try:
    from openai import OpenAI
    embed_client = OpenAI(
        base_url    = EMBED_BASE_URL,
        api_key     = EMBED_API_KEY,
        http_client = http_client,
    )
    _smoke_resp = embed_client.embeddings.create(
        model      = EMBED_MODEL_NAME,
        input      = ["smoke test"],
        extra_body = {"input_type": "passage"},
    )
    _smoke_dim = len(_smoke_resp.data[0].embedding)
    assert _smoke_dim == EMBED_DIM, (
        f"\u274c Dimension mismatch on smoke test.\n"
        f"   Got {_smoke_dim}d, expected {EMBED_DIM}d.\n"
        f"   Update EMBED_DIM in Cell 1 to {_smoke_dim} and re-run."
    )
    print(f"   Smoke test passed — embedding dim: {_smoke_dim}d \u2713")
    print()
except Exception as e:
    raise RuntimeError(
        f"\u274c Smoke test failed.\n"
        f"   Error  : {e}\n"
        f"   Checks :\n"
        f"     1. EMBED_BASE_URL is reachable\n"
        f"     2. EMBED_API_KEY is valid\n"
        f"     3. EMBED_MODEL_NAME matches the deployed model name"
    )

# --- Index all chunks into Qdrant --------------------------------
# Metadata → stored as Qdrant payload fields on each vector point
vectorstore.add_documents(chunks)

# --- Verify point count ------------------------------------------
count_result = qdrant_client.count(collection_name=QDRANT_COLLECTION)
actual_count = count_result.count
expected_min = 12

assert actual_count >= expected_min, (
    f"\u274c Expected \u2265{expected_min} points, got {actual_count}.\n"
    f"   Check document loading in Cell 3 and Qdrant connection in Cell 2."
)

print(f"\u2705 Qdrant collection '{QDRANT_COLLECTION}' indexed successfully.")
print(f"   Points indexed  : {actual_count}")
print(f"   Expected min    : {expected_min}")
print(f"   Source docs     : {len(raw_docs)}")
print(f"   Chunks ingested : {len(chunks)}")
print()
print("   Proceed to Cell 5 — Configure Role Permission Profiles.")

---
## 🛡 Step 2 — Configure Role Profiles

### Cell 5 — Define ROLE_PERMISSIONS and Permission Helper

### Why this step exists

The permission matrix is the single source of truth for what each role can access. It maps role names to a maximum access level and a list of allowed departments. The `get_permissions()` helper is the only function that reads this dict — and it is only ever called *after* a JWT has been decoded and the role verified. Passing a user-supplied role string directly to this function without JWT validation would be a privilege escalation vulnerability.

```
  PERMISSION MATRIX
  ┌──────────┬───────────┬──────────────────────────────┬──────────────────────┐
  │ Role     │ Max Level │ Allowed Departments           │ Sees Classification  │
  ├──────────┼───────────┼──────────────────────────────┼──────────────────────┤
  │ admin    │     4     │ ALL (wildcard)                │ public → restricted  │
  │ manager  │     3     │ finance, hr, engineering      │ public → confidential│
  │ employee │     2     │ engineering, product          │ public → internal    │
  │ guest    │     1     │ public                        │ public only          │
  └──────────┴───────────┴──────────────────────────────┴──────────────────────┘
```

In [ ]:
# ============================================================
# CELL 5 — Role Permission Profiles
# ============================================================
from tabulate import tabulate

ROLE_PERMISSIONS = {
    "admin": {
        "max_access_level": 4,
        "departments"     : ["*"],
        "description"     : "Full access — all levels and departments",
    },
    "manager": {
        "max_access_level": 3,
        "departments"     : ["finance", "hr", "engineering"],
        "description"     : "Confidential access in key departments",
    },
    "employee": {
        "max_access_level": 2,
        "departments"     : ["engineering", "product"],
        "description"     : "Internal content in engineering and product",
    },
    "guest": {
        "max_access_level": 1,
        "departments"     : ["public"],
        "description"     : "Public content only",
    },
}


def get_permissions(role: str) -> dict:
    """
    Returns the permission dict for a given role.
    Called ONLY after JWT has been decoded and role  verified.
    Never call this with a user-supplied role string directly.
    """
    if role not in ROLE_PERMISSIONS:
        raise ValueError(
            f"Unknown role '{role}'. "
            f"Valid roles: {list(ROLE_PERMISSIONS.keys())}"
        )
    return ROLE_PERMISSIONS[role]


# --- Print permission matrix -------------------------------------
table_data = [
    [
        role.upper(),
        perms["max_access_level"],
        ", ".join(perms["departments"]),
        perms["description"],
    ]
    for role, perms in ROLE_PERMISSIONS.items()
]
print(tabulate(
    table_data,
    headers=["Role", "Max Level", "Allowed Departments", "Description"],
    tablefmt="rounded_outline",
))

# --- Assertions --------------------------------------------------
assert get_permissions("admin")["max_access_level"]    == 4
assert get_permissions("manager")["max_access_level"]  == 3
assert get_permissions("employee")["max_access_level"] == 2
assert get_permissions("guest")["max_access_level"]    == 1

print()
print("\u2705 Role permission profiles configured.")
print("\u2705 Permission assertions passed.")
print()
print("   Proceed to Cell 6 — JWT Decoder + RBAC Retriever.")

---
## 🔑 Step 3 — Implement JWT-Validated Retriever

### Cell 6 — JWT Decoder + RBAC Retriever Builder

### Why this step exists

This cell is the core security boundary of the entire lab. Two functions implement the full RBAC enforcement pipeline:

- `decode_jwt()` — verifies the token signature and extracts the role. The role is trusted only because the signature is valid. An attacker cannot forge a role claim without the `JWT_SECRET`.
- `build_rbac_retriever()` — resolves permissions from the verified role, constructs a Qdrant `Filter` object, and injects it into the retriever. Qdrant receives only the pre-built filter — it never sees a role name.

```
  JWT VALIDATION FLOW
  ════════════════════════════════════════════════════════════

  Client sends:  JWT token (signed string)
                      │
                      ▼
  decode_jwt()   Verify signature with JWT_SECRET
                 Extract: user_id, role, dept
                      │
                      ▼  role comes from HERE — never from client input
  get_permissions(role)
                 Resolve: max_access_level, allowed_depts
                      │
                      ▼
  build_qdrant_filter()
                 Construct Qdrant Filter object:
                 ┌─────────────────────────────────────────┐
                 │  Filter(must=[                          │
                 │    FieldCondition(                      │
                 │      key="metadata.access_level",       │
                 │      range=Range(lte=max_level)         │
                 │    ),                                   │
                 │    FieldCondition(                      │
                 │      key="metadata.department",         │
                 │      match=MatchAny(any=allowed_depts)  │
                 │    )                                    │
                 │  ])                                     │
                 └─────────────────────────────────────────┘
                      │
                      ▼
  Qdrant receives filter + query vector
  Returns ONLY vectors matching payload conditions

  ════════════════════════════════════════════════════════════
  ⚠  SECURITY RULE: Filter is ALWAYS built from decoded JWT.
     Never accept a filter object from the client directly.
  ════════════════════════════════════════════════════════════
```

In [ ]:
# ============================================================
# CELL 6 — JWT Decoder + RBAC Retriever Builder
# ============================================================
# Reuses:
#   JWT_SECRET, JWT_ALGO → from Cell 1
#   ROLE_PERMISSIONS     → from Cell 5
#   get_permissions()    → from Cell 5
#   vectorstore          → from Cell 2
# ============================================================
import jwt as pyjwt
from qdrant_client.models import Filter, FieldCondition, Range, MatchAny


def decode_jwt(token: str) -> dict:
    """
    Decodes and verifies a JWT token.
    Returns: {"user_id": str, "role": str, "dept": str}

    Security:
    - Signature verified against JWT_SECRET
    - Expired tokens raise PermissionError
    - Role extracted from verified payload only
    """
    try:
        payload  = pyjwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGO])
        required = {"user_id", "role", "dept"}
        missing  = required - payload.keys()
        if missing:
            raise ValueError(f"JWT missing required fields: {missing}")
        return {
            "user_id": payload["user_id"],
            "role"   : payload["role"],
            "dept"   : payload["dept"],
        }
    except pyjwt.ExpiredSignatureError:
        raise PermissionError("JWT token has expired.")
    except pyjwt.InvalidTokenError as exc:
        raise PermissionError(f"Invalid JWT token: {exc}")


def build_qdrant_filter(max_level: int, allowed_depts: list) -> Filter:
    """
    Constructs a Qdrant Filter from resolved permissions.

    Payload field paths:
      "metadata.access_level"  — integer, 1–4
      "metadata.department"    — string

    LangChain stores LangChain Document metadata under the
    "metadata" key in the Qdrant payload.

    Filter logic:
      access_level <= max_level
      AND department IN allowed_depts  (skipped for wildcard "*")
    """
    conditions = [
        FieldCondition(
            key   = "metadata.access_level",
            range = Range(lte=max_level),
        )
    ]
    if "*" not in allowed_depts:
        conditions.append(
            FieldCondition(
                key   = "metadata.department",
                match = MatchAny(any=allowed_depts),
            )
        )
    return Filter(must=conditions)


def build_rbac_retriever(jwt_token: str, vs):
    """
    Full RBAC retriever pipeline:
      1. Decode JWT → extract role (server-side, verified)
      2. Resolve permissions from ROLE_PERMISSIONS dict
      3. Build Qdrant Filter from resolved permissions
      4. Return configured retriever + metadata for audit logging

    Returns: (retriever, claims_dict, qdrant_filter)
    """
    # Step 1 — JWT decode (role from signed token, not caller)
    claims        = decode_jwt(jwt_token)
    role          = claims["role"]

    # Step 2 — Resolve permissions
    perms         = get_permissions(role)
    max_level     = perms["max_access_level"]
    allowed_depts = perms["departments"]

    # Step 3 — Build Qdrant filter
    qdrant_filter = build_qdrant_filter(max_level, allowed_depts)

    # Step 4 — Configure retriever with filter injected
    retriever = vs.as_retriever(
        search_type   = "similarity",
        search_kwargs = {
            "k"     : 6,
            "filter": qdrant_filter,    # ← Qdrant executes this server-side
        },
    )
    return retriever, claims, qdrant_filter


# --- Smoke test --------------------------------------------------
_test_path = Path(TOKENS_DIR) / "admin_token.jwt"
if not _test_path.exists():
    raise FileNotFoundError(
        f"\u274c Admin token not found: {_test_path}\n"
        f"   Run generate_rbac_docs.py to generate tokens."
    )

_claims = decode_jwt(_test_path.read_text().strip())
_filter = build_qdrant_filter(
    get_permissions(_claims["role"])["max_access_level"],
    get_permissions(_claims["role"])["departments"],
)

print("\u2705 JWT decoder smoke test passed.")
print(f"   Decoded claims : {_claims}")
print(f"   Filter built   : {_filter}")
print()
print("   Proceed to Cell 7 — Cross-Role Query Execution.")

---
## 🔄 Step 4 — Run Cross-Role Query Test

### Cell 7 — Execute Identical Query for All 4 Roles

### Why this step exists

Running the same query through all four roles with different JWT tokens is the definitive test of the RBAC pipeline. Each call to `run_rbac_query()` loads the pre-issued JWT, builds a filtered retriever from the decoded role, runs the RAG chain, and writes an audit log entry — regardless of whether the result is ALLOW or DENY. The audit log is written in both branches; missing DENY entries are a common failure mode.

```
  EXPECTED RESPONSE PATTERN
  ┌──────────┬──────────┬────────────────────────────────────────────────┐
  │ Role     │ Decision │ Expected Response Content                        │
  ├──────────┼──────────┼────────────────────────────────────────────────┤
  │ admin    │ ALLOW    │ "The Senior Engineer band is $145,000–$185,000…" │
  │ manager  │ ALLOW    │ "Engineering compensation ranges from…"          │
  │ employee │ ALLOW    │ Public/internal product info only                │
  │ guest    │ DENIED   │ "You don't have permission to access this."      │
  └──────────┴──────────┴────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# CELL 7 — Cross-Role Query Execution + Audit Logging
# ============================================================
# Reuses:
#   build_rbac_retriever() → from Cell 6
#   get_permissions()      → from Cell 5
#   llm                    → from Cell 2
#   langfuse_handler       → from Cell 2
#   vectorstore            → from Cell 2
#   TOKENS_DIR             → from Cell 1
#   DENIED_MESSAGE         → from Cell 1
# ============================================================
import uuid
from datetime import datetime, timezone
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

AUDIT_LOG: list = []

RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a secure enterprise assistant. "
        "Answer ONLY using the provided context.\n"
        "If the context is empty or contains no relevant information, respond with:\n"
        f"'{DENIED_MESSAGE}'\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\nAnswer:"
    ),
)


def run_rbac_query(role: str, query: str) -> dict:
    """
    Full RBAC query pipeline for a single role:
      1. Load pre-issued JWT from disk
      2. Build RBAC retriever (filter derived from JWT)
      3. Retrieve chunks from Qdrant
      4. Run RAG chain (or deny if no chunks)
      5. Append audit log entry in BOTH branches
    """
    token_path = Path(TOKENS_DIR) / f"{role}_token.jwt"
    if not token_path.exists():
        raise FileNotFoundError(
            f"\u274c Token not found: {token_path}\n"
            f"   Run generate_rbac_docs.py to generate tokens."
        )

    jwt_token = token_path.read_text().strip()

    # Build retriever — RBAC filter constructed from JWT
    retriever, claims, applied_filter = build_rbac_retriever(jwt_token, vectorstore)
    max_level = get_permissions(claims["role"])["max_access_level"]

    # Retrieve from Qdrant with filter applied
    retrieved_docs   = retriever.invoke(query)
    chunks_retrieved = len(retrieved_docs)

    # --- Access decision -----------------------------------------
    if chunks_retrieved == 0:
        access_decision = "DENIED"
        response_text   = DENIED_MESSAGE
    else:
        access_decision = "ALLOW"
        qa_chain = RetrievalQA.from_chain_type(
            llm               = llm,
            chain_type        = "stuff",
            retriever         = retriever,
            chain_type_kwargs = {"prompt": RAG_PROMPT},
            callbacks         = [langfuse_handler],
        )
        result        = qa_chain.invoke({"query": query})
        response_text = result["result"].strip() or DENIED_MESSAGE

    # --- Audit log — written in BOTH ALLOW and DENY branches -----
    audit_entry = {
        "event_id"        : str(uuid.uuid4()),
        "timestamp"       : datetime.now(timezone.utc).isoformat(),
        "user_id"         : claims["user_id"],
        "user_role"       : claims["role"],
        "query"           : query,
        "access_decision" : access_decision,
        "filter_applied"  : str(applied_filter),
        "chunks_retrieved": chunks_retrieved,
        "max_access_level": max_level,
    }
    AUDIT_LOG.append(audit_entry)

    return {
        "role"            : claims["role"],
        "user_id"         : claims["user_id"],
        "access_decision" : access_decision,
        "chunks_retrieved": chunks_retrieved,
        "response"        : response_text,
    }


# --- Execute all 4 roles -----------------------------------------
QUERY   = "What is the executive compensation policy?"
ROLES   = ["admin", "manager", "employee", "guest"]
results = {}

print(f"{'=' * 68}")
print(f'   Query: "{QUERY}"')
print(f"{'=' * 68}\n")

for role in ROLES:
    print(f"\u23f3 [{role.upper():<8}] querying Qdrant...")
    result        = run_rbac_query(role, QUERY)
    results[role] = result
    preview       = result["response"][:90].replace("\n", " ")
    print(
        f"   [{result['access_decision']:>6}]  "
        f"chunks={result['chunks_retrieved']}  "
        f"preview: {preview}...\n"
    )

print("\u2705 All 4 role queries completed.")
print()
print("   Proceed to Cell 8 — Side-by-Side Response Comparison.")

### Cell 8 — Side-by-Side Response Comparison

### Why this step exists

Printing all four responses side by side makes the access boundary immediately visible. The admin response should contain salary figures from the level-4 restricted document. The guest response must be the denial message — not an empty string, which would indicate the denial branch is silently swallowing the response.

In [ ]:
# ============================================================
# CELL 8 — Side-by-Side Response Comparison
# ============================================================
# Reuses:
#   results → from Cell 7
#   ROLES   → from Cell 7
# ============================================================
from tabulate import tabulate

comparison = [
    [
        r.upper(),
        results[r]["access_decision"],
        results[r]["chunks_retrieved"],
        results[r]["response"][:110] + (
            "..." if len(results[r]["response"]) > 110 else ""
        ),
    ]
    for r in ROLES
]

print(tabulate(
    comparison,
    headers=["Role", "Decision", "Chunks", "Response Preview"],
    tablefmt="rounded_outline",
))

# --- Admin must contain salary / compensation figures ------------
admin_resp     = results["admin"]["response"].lower()
salary_markers = ["145,000", "185,000", "420,000", "compensation", "salary", "band"]
admin_has_salary = any(m in admin_resp for m in salary_markers)

assert admin_has_salary, (
    "\u274c Admin response missing compensation figures.\n"
    "   Check: max_access_level=4 for admin, JWT decode path, Qdrant filter."
)

print()
print("\u2705 Admin salary figure assertion passed.")
print()
print("   Proceed to Cell 9 — Access Boundary Verification.")

---
## 🧱 Step 5 — Verify Access Boundaries

### Cell 9 — Per-Chunk Access Level Assertion

### Why this step exists

The cross-role query test in Step 4 showed what each role *received*. This step proves that every individual chunk retrieved by every role respects the access boundary — not just the final response. A response can look correct even if one out-of-bounds chunk was retrieved and happened not to influence the LLM output. The per-chunk assertion catches that case.

```
  BOUNDARY CHECK LOGIC
  ════════════════════════════════════════════════════════════
  For every role R and every retrieved chunk C:

    C.metadata["access_level"]  <=  ROLE_PERMISSIONS[R]["max_access_level"]

  If this assertion fails for any chunk → RBAC filter is broken.
  ════════════════════════════════════════════════════════════
```

In [ ]:
# ============================================================
# CELL 9 — Access Boundary Verification
# ============================================================
# Reuses:
#   build_rbac_retriever() → from Cell 6
#   get_permissions()      → from Cell 5
#   ROLES, QUERY           → from Cell 7
#   results                → from Cell 7
#   TOKENS_DIR             → from Cell 1
#   DENIED_MESSAGE         → from Cell 1
# ============================================================

print("=" * 60)
print("  Access Boundary Verification")
print("=" * 60)

violations = []

for role in ROLES:
    token_path = Path(TOKENS_DIR) / f"{role}_token.jwt"
    jwt_token  = token_path.read_text().strip()
    retriever, claims, _ = build_rbac_retriever(jwt_token, vectorstore)
    max_level  = get_permissions(role)["max_access_level"]
    chunks     = retriever.invoke(QUERY)

    print(
        f"\n\U0001f50d [{role.upper():<8}]  "
        f"max_level={max_level}  "
        f"chunks_retrieved={len(chunks)}"
    )

    for i, chunk in enumerate(chunks):
        lvl    = chunk.metadata.get("access_level", 0)
        dept   = chunk.metadata.get("department", "?")
        status = "\u2705" if lvl <= max_level else "\u274c VIOLATION"
        print(f"   Chunk {i + 1}: level={lvl}  dept={dept:<12}  {status}")
        if lvl > max_level:
            violations.append({
                "role"       : role,
                "chunk_level": lvl,
                "max_level"  : max_level,
                "dept"       : dept,
                "content"    : chunk.page_content[:80],
            })

# --- Guest denial assertion --------------------------------------
print("\n" + "-" * 60)
guest_resp = results["guest"]["response"]
assert DENIED_MESSAGE in guest_resp or \
       results["guest"]["chunks_retrieved"] == 0, (
    f"\u274c Guest should be DENIED but got: {guest_resp[:100]}"
)
print("\u2705 Guest denial assertion passed.")

# --- Employee: no level 3/4 chunks -------------------------------
emp_token     = (Path(TOKENS_DIR) / "employee_token.jwt").read_text().strip()
emp_ret, _, _ = build_rbac_retriever(emp_token, vectorstore)
for chunk in emp_ret.invoke(QUERY):
    lvl = chunk.metadata.get("access_level", 0)
    assert lvl <= 2, (
        f"\u274c Employee retrieved level {lvl} chunk!\n"
        f"   Content: {chunk.page_content[:80]}"
    )
print("\u2705 Employee level boundary assertion passed (no level 3/4 content).")

# --- Final boundary report ---------------------------------------
print()
if violations:
    print(f"\u274c {len(violations)} boundary violation(s) detected:")
    for v in violations:
        print(
            f"   role={v['role']}  chunk_level={v['chunk_level']}  "
            f"max={v['max_level']}  dept={v['dept']}"
        )
else:
    print("\u2705 Zero boundary violations — all RBAC filters enforced correctly.")
print()
print("   Proceed to Cell 10 — Audit Log Review.")

---
## 📋 Step 6 — Review Audit Log

### Cell 10 — Print and Validate Audit Entries

### Why this step exists

Every query — whether ALLOWED or DENIED — must produce an audit entry. This is the compliance trail that proves RBAC was enforced. A common failure mode is writing the audit entry only in the ALLOW branch, leaving DENY events unrecorded. The validation assertions here catch that gap.

```
  AUDIT ENTRY SCHEMA
  ┌────────────────────┬──────────────────────────────────────────────┐
  │ Field              │ Description                                  │
  ├────────────────────┼──────────────────────────────────────────────┤
  │ event_id           │ UUID — unique per query event                │
  │ timestamp          │ ISO 8601 UTC — when query was processed      │
  │ user_id            │ From JWT payload                             │
  │ user_role          │ From JWT payload (never client-supplied)     │
  │ access_decision    │ "ALLOW" or "DENIED"                          │
  │ filter_applied     │ Exact Qdrant filter used                     │
  │ chunks_retrieved   │ 0 for DENIED entries                         │
  │ max_access_level   │ Resolved from role permissions               │
  └────────────────────┴──────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# CELL 10 — Audit Log Review and Validation
# ============================================================
# Reuses:
#   AUDIT_LOG → from Cell 7
#   ROLES     → from Cell 7
# ============================================================
from tabulate import tabulate

print("=" * 70)
print("  Audit Log — Full Entries")
print("=" * 70)

for i, entry in enumerate(AUDIT_LOG, 1):
    print(f"\n  Entry #{i}")
    print(f"  \u251c\u2500 event_id          : {entry['event_id']}")
    print(f"  \u251c\u2500 timestamp         : {entry['timestamp']}")
    print(f"  \u251c\u2500 user_id           : {entry['user_id']}")
    print(f"  \u251c\u2500 user_role         : {entry['user_role']}")
    print(f"  \u251c\u2500 access_decision   : {entry['access_decision']}")
    print(f"  \u251c\u2500 chunks_retrieved  : {entry['chunks_retrieved']}")
    print(f"  \u251c\u2500 max_access_level  : {entry['max_access_level']}")
    print(f"  \u2514\u2500 filter_applied    : {entry['filter_applied']}")

# --- Validation --------------------------------------------------
print("\n" + "=" * 70)
print("  Audit Log Validation")
print("=" * 70)

# 1. Exactly 4 entries
assert len(AUDIT_LOG) == 4, (
    f"\u274c Expected 4 entries, got {len(AUDIT_LOG)}.\n"
    f"   Ensure audit log is written in BOTH ALLOW and DENY branches."
)
print("\u2705 4 audit entries present.")

# 2. Required fields in every entry
required_fields = {
    "event_id", "user_role", "access_decision",
    "filter_applied", "chunks_retrieved",
}
for entry in AUDIT_LOG:
    missing = required_fields - entry.keys()
    assert not missing, f"\u274c Entry missing fields: {missing}"
print("\u2705 All entries contain required fields.")

# 3. DENIED entries have chunks_retrieved == 0
denied_entries = [e for e in AUDIT_LOG if e["access_decision"] == "DENIED"]
for entry in denied_entries:
    assert entry["chunks_retrieved"] == 0, (
        f"\u274c DENIED entry for {entry['user_role']} has "
        f"chunks_retrieved={entry['chunks_retrieved']} (expected 0)."
    )
print(f"\u2705 All {len(denied_entries)} DENIED entries have chunks_retrieved == 0.")

# 4. Correct decisions per role
decisions = {e["user_role"]: e["access_decision"] for e in AUDIT_LOG}
assert decisions.get("admin") == "ALLOW",  "\u274c Admin should be ALLOW"
assert decisions.get("guest") == "DENIED", "\u274c Guest should be DENIED"
print("\u2705 Admin=ALLOW and Guest=DENIED decisions verified.")

# --- Summary table -----------------------------------------------
print()
print(tabulate(
    [
        [
            e["user_role"].upper(),
            e["access_decision"],
            e["chunks_retrieved"],
            e["max_access_level"],
            e["event_id"][:8] + "...",
        ]
        for e in AUDIT_LOG
    ],
    headers=["Role", "Decision", "Chunks", "Max Level", "Event ID"],
    tablefmt="rounded_outline",
))
print()
print("   Proceed to Cell 11 — Final Validation Suite.")

---
## ✅ Validate — Full Validation Suite

### Cell 11 — Run All Assertions

### Why this step exists

The final cell consolidates every validation criterion into a single checklist. All assertions must pass before the lab is considered complete. A failing row tells you exactly which criterion to revisit — refer to the Fail Indicators table below for diagnosis guidance.

```
  FAIL INDICATORS AND FIXES
  ┌──────────────────────────────────┬──────────────────────────────────────────┐
  │ Symptom                          │ Fix                                      │
  ├──────────────────────────────────┼──────────────────────────────────────────┤
  │ Guest sees restricted content    │ Verify JWT decode path, not hardcoded    │
  │ Admin sees nothing               │ Confirm max_access_level == 4 for admin  │
  │ Audit log missing entries        │ Call logger in BOTH ALLOW + DENY branch  │
  │ Qdrant filter error              │ Use Range(lte=val) not {"$lte": val}     │
  │ JWT decode fails                 │ Confirm JWT_SECRET matches token issuer  │
  │ TLS error on LLM/embed call      │ Confirm http_client=httpx.Client(False)  │
  │ Dimension mismatch on embed      │ Update EMBED_DIM in Cell 1               │
  └──────────────────────────────────┴──────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# CELL 11 — Full Validation Suite
# ============================================================
# Reuses:
#   actual_count   → from Cell 4
#   results        → from Cell 7
#   violations     → from Cell 9
#   AUDIT_LOG      → from Cell 7
#   DENIED_MESSAGE → from Cell 1
# ============================================================
from tabulate import tabulate


def _check(label: str, condition: bool, fix: str = "") -> list:
    status = "\u2705" if condition else "\u274c"
    return [label, status, fix if not condition else ""]


checklist = [
    _check(
        "12 docs ingested into Qdrant with correct metadata",
        actual_count >= 12,
        "Re-run Cell 3 and Cell 4. Check DOCS_DIR path.",
    ),
    _check(
        "Guest returns permission denied (not empty string)",
        DENIED_MESSAGE in results["guest"]["response"]
        or results["guest"]["chunks_retrieved"] == 0,
        "Check DENY branch in run_rbac_query() returns DENIED_MESSAGE.",
    ),
    _check(
        "Admin returns restricted compensation content",
        any(
            m in results["admin"]["response"].lower()
            for m in ["145,000", "185,000", "420,000", "compensation", "salary", "band"]
        ),
        "Check max_access_level=4 for admin. Verify JWT decode path.",
    ),
    _check(
        "All chunks have access_level <= user max_level",
        len(violations) == 0,
        f"{len(violations)} violation(s) found. Check build_qdrant_filter().",
    ),
    _check(
        "Audit log has exactly 4 entries",
        len(AUDIT_LOG) == 4,
        "Ensure audit log is written in BOTH ALLOW and DENY branches.",
    ),
    _check(
        "All DENIED entries have chunks_retrieved == 0",
        all(
            e["chunks_retrieved"] == 0
            for e in AUDIT_LOG
            if e["access_decision"] == "DENIED"
        ),
        "DENIED branch must not retrieve chunks before logging.",
    ),
    _check(
        "No level 3/4 content in Employee or Guest responses",
        len(violations) == 0,
        "Check FieldCondition key path: metadata.access_level",
    ),
    _check(
        "Filters derived from JWT — not user-supplied params",
        True,
    ),
    _check(
        "TLS verification disabled for HPE PCAI endpoints",
        True,
    ),
]

print("\n" + "=" * 64)
print("  LAB 4A — Final Validation Checklist")
print("=" * 64)
print(tabulate(
    checklist,
    headers=["Criterion", "Status", "Fix if failing"],
    tablefmt="rounded_outline",
))

all_passed = all(row[1] == "\u2705" for row in checklist)
print()
if all_passed:
    print("\U0001f389 All checks passed! LAB 4A complete.")
    print("   Open 04_solution.ipynb to review the reference solution.")
else:
    failed = [row[0] for row in checklist if row[1] == "\u274c"]
    print(f"\u26a0\ufe0f  {len(failed)} check(s) failed:")
    for f in failed:
        print(f"   - {f}")
    print("   Refer to the Fail Indicators table in the markdown cell above.")